# فصل ۱۰: حافظه و نگهداری تاریخچه (Memory & History)
## بخش ۱: مدیریت تاریخچه گفت‌وگو (History Management)

### مفاهیم کلیدی:
1. **بی‌حالت بودن مدل‌های زبانی (Statelessness):** مدل‌های زبانی هیچ حافظه‌ای از پرامپت‌ها و تعاملات قبلی ندارند. هر درخواست به مدل کاملاً مستقل از درخواست‌های قبلی پردازش می‌شود.
2. **مکانیزم حافظه:** برای اینکه مدل زمینه (Context) گفتگو را به خاطر بسپارد، وظیفه ماست که تاریخچه پیام‌های قبلی را در قالب یک «دفترچه یادداشت» جمع‌آوری کرده و در هر تعامل جدید مجدداً برای مدل ارسال کنیم.
3. **ابزار `ChatMessageHistory`:** ساده‌ترین و پایه‌ای‌ترین روش ذخیره تاریخچه درون حافظه موقت (In-Memory) در LangChain است.

In [25]:
import os
from dotenv import load_dotenv

# بارگذاری متغیرهای محیطی
load_dotenv()

# وارد کردن کلاس‌های مربوط به مدیریت تاریخچه و انواع پیام‌ها
try:
    from langchain_community.chat_message_histories import ChatMessageHistory
except ImportError:
    from langchain_core.chat_history import InMemoryChatMessageHistory as ChatMessageHistory

from langchain_core.messages import HumanMessage, AIMessage

# مقداردهی اولیه مدل (می‌توانید از مدل جمینای یا مدل‌های سازگار با OpenAI استفاده کنید)
from langchain_google_genai import ChatGoogleGenerativeAI

chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)

print("✅ کتابخانه‌ها و مدل با موفقیت لود شدند!")

✅ کتابخانه‌ها و مدل با موفقیت لود شدند!


### ۱. آشنایی با عملکرد `ChatMessageHistory`

کلاس `ChatMessageHistory` مانند یک لیست هوشمند عمل می‌کند که پیام‌های کاربر (`HumanMessage`) و پیام‌های مدل (`AIMessage`) را به ترتیب اضافه و نگهداری می‌کند.

In [26]:
# ساخت نمونه جدید از تاریخچه
history = ChatMessageHistory()

# اضافه کردن پیام کاربر و سپس پیام هوش مصنوعی به تاریخچه
history.add_user_message("این جمله رو به فرانسوی ترجمه کن: من عاشق برنامه‌نویسی‌ام")
history.add_ai_message("J'adore la programmation.")

# مشاهده پیام‌های ثبت‌شده در تاریخچه
print("محتویات تاریخچه:")
for msg in history.messages:
    print(f"[{type(msg).__name__}]: {msg.content}")

محتویات تاریخچه:
[HumanMessage]: این جمله رو به فرانسوی ترجمه کن: من عاشق برنامه‌نویسی‌ام
[AIMessage]: J'adore la programmation.


### ۲. ساخت یک چت‌بات تعاملی با حفظ تاریخچه

در هر دور از مکالمه، ۴ مرحله زیر را انجام می‌دهیم:
1. پیام جدید را از کاربر دریافت می‌کنیم.
2. پیام را به `chat_history` اضافه می‌کنیم.
3. کل `chat_history.messages` را به مدل ارسال می‌کنیم.
4. پاسخ مدل را به صورت استریم (Stream) نمایش داده و سپس به `chat_history` اضافه می‌کنیم.

> برای خروج از مکالمه، عدد `0` را وارد کنید.

In [24]:
print("شروع چت (برای خروج عدد 0 را وارد کنید):")
chat_history = ChatMessageHistory()

while True:
    user_input = input("User: ")
    print(f"User: {user_input}")
    
    if user_input.strip() == "0":
        print("پایان مکالمه.")
        break
        
    # ۱. افزودن پیام کاربر به تاریخچه
    chat_history.add_user_message(user_input)
    
    # ۲. ارسال تاریخچه به مدل و استریم پاسخ
    print("AI: ", end="", flush=True)
    ai_response = ""
    for chunk in chat_model.stream(chat_history.messages):
        content = chunk.text  # 👈 فقط این خط تغییر کرده (text به جای content)
        print(content, end="", flush=True)
        ai_response += content
    print()
    
    # ۳. ثبت پاسخ مدل در تاریخچه برای دورهای بعدی
    chat_history.add_ai_message(ai_response)

شروع چت (برای خروج عدد 0 را وارد کنید):
User: تا الان چت ما در مورد چی بود
AI: تا الان چت خاصی با هم نداشتیم! این اولین پیام شماست. 

هر زمان که دوست داشتید، می‌توانید بحث را شروع کنید؛ چه درباره یک سوال علمی باشد، چه برنامه‌نویسی، نوشتن متن، یا هر موضوع دیگری که دوست دارید در موردش با هم گفتگو کنیم. چطور می‌توانم کمکتان کنم؟
User: خب بگو ببینم در مورد گرایش نرم افزار دانشگاه خوارزمی و اساتیدش در مقطع کارشناسی ارشد چی میدونی
AI: دانشگاه خوارزمی (به‌ویژه پردیس کرج و دانشکده ریاضی و کامپیوتر) یکی از دانشگاه‌های قدیمی و معتبر کشور است و گروه مهندسی کامپیوتر آن در مقطع کارشناسی ارشد (به‌خصوص گرایش نرم‌افزار) جایگاه خوبی دارد. 

در ادامه، شناخت نسبتاً کاملی از وضعیت گرایش نرم‌افزار ارشد در دانشگاه خوارزمی، اساتید شاخص و فضای کلی آن به شما ارائه می‌دهم:

### ۱. فضای کلی مقطع ارشد نرم‌افزار در خوارزمی
* **پژوهش‌محور بودن:** در مقطع ارشد، کیفیت تحصیل تا حد زیادی به خود دانشجو و استاد راهنما بستگی دارد. خوارزمی دانشجویان ارشد خوبی جذب می‌کند و رقابت برای گرفتن اساتید خوب وجود دارد.
* **ارتباط ب

### ۳. بررسی نقاط ضعف و چالش‌های این روش

هرچند استفاده مستقیم از `ChatMessageHistory` ساده و سریع است، اما در پروژه‌های واقعی با چالش‌های زیر روبرو می‌شود:

1. **پر شدن پنجره زمینه (Context Window Overflow):**
   با افزایش طول گفتگو، تعداد کل توکن‌ها بالا می‌رود و در نهایت از سقف مدل فراتر می‌رود و خطای طول Context می‌دهد.

2. **افزایش هزینه‌ها (Token Cost) و تاخیر (Latency):**
   چون در هر دور کل گفتگوهای قبلی از ابتدا فرستاده می‌شود، هر چه گفتگو طولانی‌تر شود، مصرف توکن ورودی و زمان انتظار پاسخ بیشتر می‌شود.

3. **عدم ماندگاری (Ephemeral / In-Memory):**
   با ریستارت شدن اسکریپت یا کربن نوت‌بوک، کل تاریخچه از بین می‌رود.

> در درسنامه‌های بعدی این فصل، روش‌های بهینه‌سازی از جمله **خلاصه‌سازی حافظه (Summarization)**، **پنجره لغزان پیام‌ها (Sliding Window)** و **ذخیره در دیتابیس پایدار (Persistent Storage)** را یاد خواهیم گرفت.